In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data" / "zenodo").exists() and (path / "experiments").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from the current working directory")

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data" / "zenodo" / "Melanoma"
OUTPUT_DIR = REPO_ROOT / "experiments" / "melanoma" / "outputs"
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import seaborn as sns

In [ ]:
from scipy import sparse
from scipy.stats import entropy, spearmanr, mannwhitneyu, fisher_exact, probplot, kruskal
from scipy.spatial import cKDTree

# H&E image

In [ ]:
img = mpimg.imread(DATA_DIR / "melanoma_rep1.png")

plt.imshow(img)
plt.axis("off")   # hides axes
plt.show()

# Load data ($k=4$)

In [ ]:
adata = sc.read_h5ad(DATA_DIR / "ST_mel1_rep2.h5ad")
adata.var_names_make_unique()

sc.pp.filter_genes(adata, min_counts=11)

adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata)
adata.layers["norm"] = adata.X.copy()
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()

adata

In [ ]:
SIGNATURES = {
    "Tumor cells (center)": ["SPP1","HSPB1","ATP1A1","NDRG1","ANXA1"],
    "Tumor cells (border)": ["CXCL10","WARS1","C1QB","GBP4","CXCL9"],
    "Stroma": ["MYL9","CFD","DCN","COL3A1","CH507-513H4.5"],
    "Lymphoid": ["LTB","CD79A","MS4A1","CCL21","CD79B"],
}
marker_genes = [g for ct in SIGNATURES.values() for g in ct if g in adata.var_names]

In [ ]:
K=4

## Proportions ($H$)

In [ ]:
snmf_dir = OUTPUT_DIR / f"K{K}"
snmf_proportions = pd.read_csv(snmf_dir / "SNMF_proportions.csv", index_col=0)
snmf_proportions

In [ ]:
adata.obs = pd.concat([adata.obs, snmf_proportions], axis=1)

## Signatures ($W$)

In [ ]:
snmf_signatures = pd.read_csv(snmf_dir / "signatures.csv", index_col=0)
snmf_signatures

In [ ]:
adata.var = pd.concat([adata.var, snmf_signatures], axis=1)

## Dispersion parameters ($\phi, \alpha, \beta$)

In [ ]:
phi = pd.read_csv(snmf_dir / "phi.csv", index_col=0).T.loc[adata.obs_names, adata.var_names]
alpha = pd.read_csv(snmf_dir / "alpha.csv", index_col=0).T.loc[adata.var_names].values
beta = pd.read_csv(snmf_dir / "beta.csv", index_col=0).T.loc[adata.obs_names].values

adata.var["alpha"] = alpha
adata.obs["beta"] = beta

adata.layers["phi"] = phi
adata.layers["theta"] = 1 / phi

## Spatial filtering matrix $S$

In [ ]:
TAU = 0.8

In [ ]:
from scipy.spatial.distance import cdist
from scipy.optimize import minimize

positions = np.array(adata.obs_names.str.split("x").tolist()).astype(float)

x = positions[:, 0]
y = positions[:, 1]

def mean_value(gamma, x, y, tau):
    gamma = gamma[0] if np.ndim(gamma) else gamma

    # Pairwise squared Euclidean distances
    coords = np.column_stack((x, y))
    D2 = cdist(coords, coords, metric='sqeuclidean')

    # Similarity matrix
    S = np.exp(-gamma * D2)
    S[S < 1e-3] = 0

    # Row normalization
    row_sums = S.sum(axis=1, keepdims=True)
    S = S / row_sums

    # Objective function
    return (np.mean(np.diag(S)) - tau) ** 2

# Optimize gamma
result = minimize(
    mean_value,
    x0=[1.0],
    args=(x, y, TAU),
    method="BFGS"
)

gamma = result.x[0]

# Compute final normalized similarity matrix
coords = np.column_stack((x, y))
D2 = cdist(coords, coords, metric='sqeuclidean')

S = np.exp(-gamma * D2)
S[S < 1e-3] = 0

row_sums = S.sum(axis=1, keepdims=True)
S = S / row_sums

## Inferred matrix $\hat{V}=WHS$

In [ ]:
V = (snmf_signatures.values @ snmf_proportions.values.T @ S).T
adata.X = V

adata.layers["inferred_counts"] = adata.X.copy()
sc.pp.normalize_total(adata)
adata.layers["inferred_norm"] = adata.X.copy()
sc.pp.log1p(adata)
adata.layers["inferred_lognorm"] = adata.X.copy()

# Functions

In [ ]:
def valid_genes(genes):
    return [g for g in genes if g in adata.var_names]

def add_expr_score(genes, key, layer=None):
    genes = valid_genes(genes)
    if layer:
        X = adata[:, genes].layers[layer]
    else:
        X = adata[:, genes].X
    vals = X.mean(axis=1)
    adata.obs[key] = vals
    return genes

def add_phi_score(genes, key):
    genes = [g for g in genes if g in phi.columns]
    adata.obs[key] = phi[genes].mean(axis=1)
    return genes

def zscore(s):
    s = pd.Series(s).astype(float)
    sd = s.std()
    return (s - s.mean()) / (sd if sd and not np.isnan(sd) else 1.0)

def compare_by_mask(value_key, mask):
    x = adata.obs.loc[mask, value_key].dropna()
    y = adata.obs.loc[~mask, value_key].dropna()

    if len(x) and len(y):
        res = mannwhitneyu(x, y, alternative="two-sided")
        U = res.statistic
        p = res.pvalue

        # Rank-biserial correlation
        # Positive -> values tend to be higher in x than in y
        r_rb = 2 * U / (len(x) * len(y)) - 1
    else:
        U = np.nan
        p = np.nan
        r_rb = np.nan

    return pd.Series({
        "feature": value_key,
        "in_median": x.median(),
        "out_median": y.median(),
        "U": U,
        "p": p,
        "effect_size_r": r_rb,
        "n_in": len(x),
        "n_other": len(y),
    })

# Annotation

In [ ]:
import numpy as np

snmf_z = (snmf_signatures - snmf_signatures.mean(axis=0)) / snmf_signatures.std(axis=0)

score_table = pd.DataFrame(index=SIGNATURES.keys(),
                           columns=snmf_z.columns,
                           dtype=float)

for comp in snmf_z.columns:
    for celltype, markers in SIGNATURES.items():
        valid = [g for g in markers if g in snmf_z.index]
        if len(valid) > 0:
            score_table.loc[celltype, comp] = (
                snmf_z.loc[valid, comp].mean() - snmf_z[comp].mean()
            )
        else:
            score_table.loc[celltype, comp] = -np.inf

# --------------------------------------------------
# 2️⃣ Annotation function
# --------------------------------------------------
def unique_assignment(score_table, threshold=0.2):
    
    # For each component, sort cell types by descending score
    ranked = {
        comp: score_table[comp].sort_values(ascending=False).index.tolist()
        for comp in score_table.columns
    }
    
    assigned_celltypes = set()
    final_assignment = {}
    
    # Sort components by their BEST score (descending)
    components_order = sorted(
        score_table.columns,
        key=lambda c: score_table[c].max(),
        reverse=True
    )
    
    for comp in components_order:
        for celltype in ranked[comp]:
            
            if score_table.loc[celltype, comp] < threshold:
                final_assignment[comp] = "ambiguous"
                break
            
            if celltype not in assigned_celltypes:
                final_assignment[comp] = celltype
                assigned_celltypes.add(celltype)
                break
        else:
            final_assignment[comp] = "ambiguous"
    
    return pd.Series(final_assignment)

# --------------------------------------------------
# 3️⃣ Apply to all components
# --------------------------------------------------
annotation = unique_assignment(score_table)

print(annotation)

In [ ]:
PALETTE = {
    "Tumor cells (center)": "black",
    "Stroma": "red",
    "Lymphoid": "yellow",
    "Tumor cells (border)": "gray",
    "ambiguous": "lightgray",   # optional
}

threshold = 0.2

# Best enrichment score for each component
best_scores = score_table.max(axis=0)

# Sort by score
best_scores = best_scores.sort_values(ascending=False)

# Match annotations to sorted components
annotation_sorted = annotation.loc[best_scores.index]

# Colors according to assigned cell type
colors = [PALETTE.get(ct, "lightgray") for ct in annotation_sorted]

fig, ax = plt.subplots(figsize=(max(8, 0.7 * len(best_scores)), 5))

x = np.arange(len(best_scores))

ax.bar(
    x,
    best_scores.values,
    color=colors,
    edgecolor="black",
    linewidth=0.8
)

ax.axhline(
    threshold,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=f"Threshold = {threshold}"
)

ax.set_xticks(x)
ax.set_xticklabels(best_scores.index, rotation=45, ha="right")
ax.set_xlabel("SNMF component")
ax.set_ylabel("Best enrichment score")
ax.set_title("Best cell-type enrichment score per SNMF component")

# Legend for cell types
from matplotlib.patches import Patch

legend_handles = [
    Patch(facecolor=color, edgecolor="black", label=celltype)
    for celltype, color in PALETTE.items()
    if celltype != "ambiguous"
]
legend_handles.append(
    plt.Line2D([0], [0], color="black", linestyle="--",
               label=f"Threshold = {threshold}")
)

ax.legend(handles=legend_handles, frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

fig.savefig(
    f"plots/K{K}_snmf_component_enrichment.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    f"plots/K{K}_snmf_component_enrichment.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
def gini_coefficient(values):
    """
    Compute the Gini coefficient of a one-dimensional non-negative array.

    Returns:
        0: equal loading across all components
        1: loading concentrated in one component
    """
    values = np.asarray(values, dtype=float)

    if np.any(values < 0):
        raise ValueError("Gini coefficient requires non-negative values.")

    if np.all(values == 0):
        return 0.0

    values = np.sort(values)
    n = len(values)

    return (
        2 * np.sum((np.arange(1, n + 1)) * values)
        / (n * values.sum())
        - (n + 1) / n
    )

# Gini specificity of each gene across all NMF components
gene_gini = snmf_signatures.apply(gini_coefficient, axis=1)

# Component with the highest loading for each gene
gene_component = snmf_signatures.idxmax(axis=1)

szymkiewicz_simpson = np.zeros((len(SIGNATURES), len(SIGNATURES))) # Szymkiewicz–Simpson coefficient

reverse_annotation = {v: k for k, v in annotation.items()}

for i, ct1 in enumerate(SIGNATURES):
    ct_marker_genes = set(SIGNATURES[ct1])

    for j, ct2 in enumerate(SIGNATURES):
        k = reverse_annotation.get(ct2)

        if k is None:
            raise KeyError(f"No NMF component found for cell type: {ct2}")

        assigned_to_k = gene_component == k

        score = (
            gene_gini[assigned_to_k]
            * snmf_signatures.loc[assigned_to_k, k]
        )

        top_genes = set(
            score.sort_values(ascending=False)
            .head(20)
            .index
        )

        szymkiewicz_simpson[i, j] = (
            len(ct_marker_genes & top_genes) / min(len(ct_marker_genes), len(top_genes))
        )

szymkiewicz_simpson = pd.DataFrame(
    szymkiewicz_simpson,
    index=SIGNATURES.keys(),
    columns=SIGNATURES.keys(),
)

plt.figure(figsize=(8, 6))
sns.heatmap(szymkiewicz_simpson, annot=True, cmap="vlag", center=0)
plt.title("")
plt.xlabel("Top genes")
plt.ylabel("Marker genes")

plt.tight_layout()

plt.savefig(f"plots/K{K}_marker_top_genes_overlap.png", dpi=300)
plt.savefig(f"plots/K{K}_marker_top_genes_overlap.pdf", dpi=300)
plt.show()

In [ ]:
N_TOP_GENES = 5

In [ ]:
top_genes_ordered = []
for comp in snmf_signatures.columns:
    top_genes = snmf_signatures[comp].sort_values(ascending=False).head(N_TOP_GENES).index
    top_genes_ordered.extend([g for g in top_genes if g not in top_genes_ordered])

# Subset and reorder genes
snmf_top = snmf_signatures.loc[top_genes_ordered]

# ----- 2️⃣ Scale genes (row-wise z-score) -----
snmf_scaled = snmf_top.sub(snmf_top.mean(axis=1), axis=0).div(snmf_top.std(axis=1), axis=0)

# ----- 3️⃣ Fancy heatmap with Seaborn -----
plt.figure(figsize=(12, max(6, len(top_genes_ordered) * 0.25)))

# sns.set_style("white")
# sns.set_context("talk")

# Optionally, order components by annotation or clustering
sns.heatmap(
    snmf_scaled,
    cmap="vlag",          # diverging colormap for z-scores
    center=0,
    linewidths=0.5,
    linecolor='gray',
    cbar_kws={"label": "Z-score"},
    xticklabels=[annotation.get(c, c) for c in snmf_scaled.columns],
    yticklabels=snmf_scaled.index
)

plt.xticks(fontsize=14)
plt.yticks(rotation=0, fontsize=12)

plt.tight_layout()
plt.savefig(f"plots/K{K}_top_genes_heatmap.png", dpi=300)
plt.savefig(f"plots/K{K}_top_genes_heatmap.pdf", dpi=300)
plt.show()

In [ ]:
for comp in snmf_signatures.columns:
    top_genes = snmf_signatures[comp].sort_values(ascending=False).head(N_TOP_GENES).index
    print(f"{annotation[comp]} variance: {snmf_scaled.loc[top_genes,comp].var()}")
    print(f"{annotation[comp]} range: {snmf_scaled.loc[top_genes,comp].max() - snmf_scaled.loc[top_genes,comp].min()}")
    print(f"{annotation[comp]} mean: {snmf_scaled.loc[top_genes,comp].mean()}")
    print()

In [ ]:
# Subset and reorder genes
snmf_mgs = snmf_signatures.loc[marker_genes]

# ----- 2️⃣ Scale genes (row-wise z-score) -----
snmf_scaled = snmf_mgs.sub(snmf_mgs.mean(axis=1), axis=0).div(snmf_mgs.std(axis=1), axis=0)

# ----- 3️⃣ Fancy heatmap with Seaborn -----
plt.figure(figsize=(12, max(6, len(marker_genes) * 0.25)))

# sns.set_style("white")
# sns.set_context("talk")

# Optionally, order components by annotation or clustering
sns.heatmap(
    snmf_scaled,
    cmap="vlag",          # diverging colormap for z-scores
    center=0,
    linewidths=0.5,
    linecolor='gray',
    cbar_kws={"label": "Z-score"},
    xticklabels=[annotation.get(c, c) for c in snmf_scaled.columns],
    yticklabels=snmf_scaled.index
)

plt.xticks(fontsize=14)
plt.yticks(rotation=0, fontsize=12)

plt.tight_layout()
plt.savefig(f"plots/K{K}_marker_genes_heatmap.png", dpi=300)
plt.savefig(f"plots/K{K}_marker_genes_heatmap.pdf", dpi=300)
plt.show()

In [ ]:
adata.obs.columns = [c if c not in annotation.index.tolist() else annotation[c] for c in adata.obs.columns]
adata.var.columns = [c if c not in annotation.index.tolist() else annotation[c] for c in adata.var.columns]

In [ ]:
adata.obs["snmf_component"] = adata.obs[[annotation[c] for c in snmf_signatures.columns]].idxmax(axis=1)
adata.obs["snmf_component"] = pd.Categorical(
    adata.obs["snmf_component"],
    categories=list(SIGNATURES.keys()),  # keep ambiguous if it exists
    ordered=True,
)

PALETTE = {
    "Tumor cells (center)": "black",
    "Stroma": "red",
    "Lymphoid": "yellow",
    "Tumor cells (border)": "gray",
}

sc.pl.spatial(
    adata,
    color="snmf_component",
    palette=PALETTE,
    spot_size=1,
    frameon=False,
    title="",
    show=False,
)

plt.savefig(f"plots/K{K}_clustering.png", dpi=300)
plt.savefig(f"plots/K{K}_clustering.pdf", dpi=300)
plt.show()

In [ ]:
expr = pd.DataFrame(
    adata[:, top_genes_ordered].layers["lognorm"].toarray(),
    index=adata.obs_names,
    columns=top_genes_ordered
).T

spot_order = adata.obs[adata.obs["snmf_component"] != "ambiguous"].sort_values("snmf_component").index
expr = expr[spot_order]

expr_scaled = expr.sub(expr.mean(axis=1), axis=0).div(expr.std(axis=1), axis=0)

cats = adata.obs["snmf_component"].cat.categories
colors = adata.uns["snmf_component_colors"]

lut = dict(zip(cats, colors))

col_colors = adata.obs["snmf_component"].map(lut)

plt.figure(figsize=(14, max(6, len(top_genes_ordered)*0.25)))

g = sns.clustermap(
    expr_scaled,
    row_cluster=False,
    col_cluster=False,
    cmap="vlag",
    center=0,
    col_colors=col_colors,
    xticklabels=False,
    yticklabels=expr_scaled.index,
    figsize=(14, max(6, len(top_genes_ordered)*0.25)),
    cbar_kws={"label": "Z-score"}
)

plt.tight_layout()

plt.savefig(f"plots/K{K}_V_expression_top_genes_heatmap.png", dpi=300)
plt.savefig(f"plots/K{K}_V_expression_top_genes_heatmap.pdf", dpi=300)
plt.show()

In [ ]:
expr = pd.DataFrame(
    adata[:, top_genes_ordered].layers["inferred_lognorm"].toarray(),
    index=adata.obs_names,
    columns=top_genes_ordered
).T

spot_order = adata.obs[adata.obs["snmf_component"] != "ambiguous"].sort_values("snmf_component").index
expr = expr[spot_order]

expr_scaled = expr.sub(expr.mean(axis=1), axis=0).div(expr.std(axis=1), axis=0)

cats = adata.obs["snmf_component"].cat.categories
colors = adata.uns["snmf_component_colors"]

lut = dict(zip(cats, colors))

col_colors = adata.obs["snmf_component"].map(lut)

plt.figure(figsize=(14, max(6, len(top_genes_ordered)*0.25)))

g = sns.clustermap(
    expr_scaled,
    row_cluster=False,
    col_cluster=False,
    cmap="vlag",
    center=0,
    col_colors=col_colors,
    xticklabels=False,
    yticklabels=expr_scaled.index,
    figsize=(14, max(6, len(top_genes_ordered)*0.25)),
    cbar_kws={"label": "Z-score"}
)

plt.tight_layout()

plt.savefig(f"plots/K{K}_WHS_expression_top_genes_heatmap.png", dpi=300)
plt.savefig(f"plots/K{K}_WHS_expression_top_genes_heatmap.pdf", dpi=300)
plt.show()

In [ ]:
corr_matrix = snmf_scaled.corr()
labels = [annotation.get(c, c) for c in snmf_scaled.columns]

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, xticklabels=labels, yticklabels=labels, cmap="vlag", center=0)
plt.title("Correlation Between SNMF Components (Top Genes)")

plt.tight_layout()

plt.savefig(f"plots/K{K}_top_genes_corr.png", dpi=300)
plt.savefig(f"plots/K{K}_top_genes_corr.pdf", dpi=300)
plt.show()

In [ ]:
import matplotlib

gene_to_celltype = {
    gene: cell_type
    for cell_type, genes in SIGNATURES.items()
    for gene in genes
    if gene in marker_genes
}

expr = pd.DataFrame(
    adata[:, marker_genes].layers["lognorm"].toarray(),
    index=adata.obs_names,
    columns=marker_genes,
).T

spot_order = (
    adata.obs.loc[adata.obs["snmf_component"] != "ambiguous"]
    .sort_values("snmf_component")
    .index
)

expr = expr.loc[:, spot_order]

# Z-score each gene
expr_scaled = expr.sub(expr.mean(axis=1), axis=0).div(
    expr.std(axis=1).replace(0, np.nan),
    axis=0,
)

# Column annotation: sNMF component
cats = adata.obs["snmf_component"].cat.categories
colors = adata.uns["snmf_component_colors"]
lut = dict(zip(cats, colors))

row_colors = pd.Series(
    [lut.get(gene_to_celltype[g], "lightgrey") for g in expr_scaled.index],
    index=expr_scaled.index,
    name="Marker source",
)

g = sns.clustermap(
    expr_scaled,
    row_cluster=False,
    col_cluster=False,
    cmap="vlag",
    center=0,
    row_colors=row_colors,
    xticklabels=False,
    yticklabels=expr_scaled.index,
    figsize=(14, max(6, len(marker_genes) * 0.35)),
    cbar_kws={"label": "Z-score"},
)

g.savefig(
    f"plots/K{K}_V_expression_marker_genes_heatmap.png",
    dpi=300,
    bbox_inches="tight",
)

g.savefig(
    f"plots/K{K}_V_expression_marker_genes_heatmap.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
gene_to_celltype = {
    gene: cell_type
    for cell_type, genes in SIGNATURES.items()
    for gene in genes
    if gene in marker_genes
}

expr = pd.DataFrame(
    adata[:, marker_genes].layers["inferred_lognorm"].toarray(),
    index=adata.obs_names,
    columns=marker_genes,
).T

spot_order = (
    adata.obs.loc[adata.obs["snmf_component"] != "ambiguous"]
    .sort_values("snmf_component")
    .index
)

expr = expr.loc[:, spot_order]

# Z-score each gene
expr_scaled = expr.sub(expr.mean(axis=1), axis=0).div(
    expr.std(axis=1).replace(0, np.nan),
    axis=0,
)

# Column annotation: sNMF component
cats = adata.obs["snmf_component"].cat.categories
colors = adata.uns["snmf_component_colors"]
lut = dict(zip(cats, colors))

row_colors = pd.Series(
    [lut.get(gene_to_celltype[g], "lightgrey") for g in expr_scaled.index],
    index=expr_scaled.index,
    name="Marker source",
)

g = sns.clustermap(
    expr_scaled,
    row_cluster=False,
    col_cluster=False,
    cmap="vlag",
    center=0,
    row_colors=row_colors,
    xticklabels=False,
    yticklabels=expr_scaled.index,
    figsize=(14, max(6, len(marker_genes) * 0.35)),
    cbar_kws={"label": "Z-score"},
)

g.savefig(
    f"plots/K{K}_WHS_expression_marker_genes_heatmap.png",
    dpi=300,
    bbox_inches="tight",
)

g.savefig(
    f"plots/K{K}_WHS_expression_marker_genes_heatmap.pdf",
    bbox_inches="tight",
)

plt.show()

# Proportions

In [ ]:
colors = annotation.unique().tolist()

fig = sc.pl.spatial(
    adata,
    color=colors,
    spot_size=1,
    frameon=False,
    cmap="magma",
    ncols=len(colors),
    return_fig=True,
    show=True
)

# Adjust spacing (important when many panels)
fig.tight_layout()

# Save in both formats
fig.savefig(f"plots/K{K}_proportions.png", dpi=300, bbox_inches="tight")
fig.savefig(f"plots/K{K}_proportions.pdf", dpi=300, bbox_inches="tight")

plt.close(fig)

In [ ]:
sc.pp.pca(
    adata,
    n_comps=30,
    svd_solver="arpack",
    layer="lognorm"
)

sc.pp.neighbors(
    adata,
    n_neighbors=15,
    n_pcs=30,
)

sc.tl.umap(adata)

fig = sc.pl.umap(
    adata,
    color=["snmf_component"],
    size=40,
    frameon=False,
    return_fig=True,
)

fig.savefig(f"plots/K{K}_V_umap_clustering.png", dpi=300, bbox_inches="tight")
fig.savefig(f"plots/K{K}_V_umap_clustering.pdf", dpi=300, bbox_inches="tight")
plt.show()

fig = sc.pl.umap(
    adata,
    color=list(SIGNATURES.keys()),
    size=40,
    ncols=4,
    frameon=False,
    return_fig=True,
)

fig.savefig(f"plots/K{K}_V_umap_proportions.png", dpi=300, bbox_inches="tight")
fig.savefig(f"plots/K{K}_V_umap_proportions.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
sc.pp.pca(
    adata,
    n_comps=30,
    svd_solver="arpack",
)

sc.pp.neighbors(
    adata,
    n_neighbors=15,
    n_pcs=30,
)

sc.tl.umap(adata)

fig = sc.pl.umap(
    adata,
    color=["snmf_component"],
    size=40,
    frameon=False,
    return_fig=True,
)

fig.savefig(f"plots/K{K}_WHS_umap_clustering.png", dpi=300, bbox_inches="tight")
fig.savefig(f"plots/K{K}_WHS_umap_clustering.pdf", dpi=300, bbox_inches="tight")
plt.show()

fig = sc.pl.umap(
    adata,
    color=list(SIGNATURES.keys()),
    size=40,
    ncols=4,
    frameon=False,
    return_fig=True,
)

fig.savefig(f"plots/K{K}_WHS_umap_proportions.png", dpi=300, bbox_inches="tight")
fig.savefig(f"plots/K{K}_WHS_umap_proportions.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from scipy.stats import pearsonr

X = adata.layers["lognorm"]

if sparse.issparse(X):
    X = X.toarray()

celltypes = list(SIGNATURES.keys())

means = {}
for ct in celltypes:
    means[ct] = X[adata.obs["snmf_component"] == ct].mean(axis=0)

corr = pd.DataFrame(index=celltypes, columns=celltypes, dtype=float)

for ct1 in celltypes:
    for ct2 in celltypes:
        r, _ = pearsonr(means[ct1], means[ct2])
        corr.loc[ct1, ct2] = r

# Plot heatmap
plt.figure(figsize=(8, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt=".3f",
    # cmap="RdBu_r",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
)

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Spatial autocorrelation for each cell type

In [ ]:
import squidpy as sq

In [ ]:
sq.gr.spatial_neighbors(
    adata,
    spatial_key="spatial",
    coord_type="generic"
)

# compute Moran's I for every cell-type proportion
sq.gr.spatial_autocorr(
    adata,
    mode="moran",
    genes=SIGNATURES.keys(),
    attr="obs",
    n_perms=1000
)

# results
moran_df = adata.uns["moranI"].copy()
moran_df = moran_df.sort_values("I", ascending=False)

print(moran_df)

In [ ]:
plt.figure(figsize=(6, max(4, 0.3 * len(moran_df))))
sns.barplot(
    data=moran_df.reset_index(),
    y="index",
    x="I",
    palette=PALETTE
)
plt.xlabel("Moran's I")
plt.ylabel("Cell type")
plt.title("Spatial autocorrelation of cell-type proportions")
plt.tight_layout()

plt.savefig(f"plots/K{K}_cell_type_morans_i.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_cell_type_morans_i.pdf", dpi=300, bbox_inches="tight")
plt.show()

# Dispersion analysis with additive factors $\alpha$ and $\beta$

## $\alpha_i$

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(alpha, bins=50, kde=True, ax=axes[0])
axes[0].set_title("Distribution of gene-wise dispersion α")
axes[0].set_xlabel("alpha")

sns.histplot(np.log1p(alpha), bins=50, kde=True, ax=axes[1])
axes[1].set_title("log1p(α)")
axes[1].set_xlabel("log1p(alpha)")

probplot(alpha.ravel(), dist="norm", plot=axes[2])
axes[2].set_title("QQ-plot of α")

plt.tight_layout()

plt.savefig(f"plots/K{K}_alpha_distribution.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_alpha_distribution.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
top_alpha_genes = (
    adata.var[["alpha"]]
    .sort_values("alpha", ascending=False)
)

top_alpha_genes.head(50).index.tolist()

### Mean expression vs $\alpha$

In [ ]:
X = adata.layers["lognorm"]

if sparse.issparse(X):
    gene_mean = np.asarray(X.mean(axis=0)).ravel()
else:
    gene_mean = X.mean(axis=0)

adata.var["mean_expression"] = gene_mean
adata.var["log1p_mean_expression"] = np.log1p(gene_mean)
adata.var["log1p_alpha"] = np.log1p(adata.var["alpha"])

plt.figure(figsize=(6, 5))
sns.scatterplot(
    data=adata.var,
    x="log1p_mean_expression",
    y="log1p_alpha",
    s=12,
    alpha=0.6
)
plt.xlabel("log1p(mean expression)")
plt.ylabel("log1p(alpha)")
plt.title("Gene mean expression vs gene-wise dispersion α")
plt.tight_layout()

plt.savefig(f"plots/K{K}_alpha_vs_mean_exp.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_alpha_vs_mean_exp.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
rho, pval = spearmanr(
    adata.var["mean_expression"],
    adata.var["alpha"],
    nan_policy="omit"
)

print(f"Spearman correlation mean expression vs alpha: rho={rho:.3f}, p={pval:.3e}")

**$\alpha$ does not merely track abundance**

### HVGs vs $\alpha$

In [ ]:
N = 2000

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N
)

adata.var["low_alpha"] = False
low_alpha_genes = top_alpha_genes.tail(N).index
adata.var.loc[low_alpha_genes, "low_alpha"] = True

hvg = set(adata.var_names[adata.var["highly_variable"]])
low_alpha = set(low_alpha_genes)

overlap = len(hvg & low_alpha)

print(f"HVG genes: {len(hvg)}")
print(f"High-alpha genes: {len(low_alpha)}")
print(f"Overlap: {overlap}")
print(f"Jaccard index: {overlap / len(hvg | low_alpha):.3f}")

In [ ]:
contingency = pd.crosstab(
    adata.var["highly_variable"],
    adata.var["low_alpha"]
)

contingency

For a 2x2 table, the null hypothesis is that the true odds ratio of the populations underlying the observations is one, and the observations were sampled from these populations under a condition: the marginals of the resulting table must equal those of the observed table. The statistic is the unconditional maximum likelihood estimate of the odds ratio

In [ ]:
oddsratio, pval = fisher_exact(contingency)
print(f"Fisher exact test: odds ratio={oddsratio:.3f}, p={pval:.3e}")

### Spatial maps of genes with higher $\alpha$

In [ ]:
sc.pl.spatial(
    adata,
    color=adata.var["alpha"].sort_values(ascending=False).head(10).index,
    cmap="inferno",
    ncols=5,
    spot_size=1,
    layer="lognorm",
)

In [ ]:
sc.pl.spatial(
    adata,
    color=adata.var["alpha"].sort_values(ascending=False).head(10).index,
    cmap="inferno",
    ncols=5,
    spot_size=1,
    layer="inferred_lognorm",
)

### GO enrichment analysis

In [ ]:
q = 0.95
alpha_thr = adata.var["alpha"].quantile(q)

high_alpha_genes = (
    adata.var_names[adata.var["alpha"] >= alpha_thr]
    .astype(str)
    .tolist()
)

background_genes = adata.var_names.astype(str).tolist()

print(f"High-alpha genes: {len(high_alpha_genes)}")
print(f"Background genes: {len(background_genes)}")

In [ ]:
import gseapy as gp

go_results = gp.enrichr(
    gene_list=high_alpha_genes,
    gene_sets=[
        "GO_Biological_Process_2023",
        "GO_Molecular_Function_2023",
        "GO_Cellular_Component_2023",
    ],
    organism="human",
    background=background_genes,
    outdir=None
)

In [ ]:
go_df = go_results.results.copy()
sig_go = go_df[go_df["Adjusted P-value"] < 0.05].copy()

top_go = sig_go.head(20).copy()
top_go["minus_log10_fdr"] = -np.log10(top_go["Adjusted P-value"])

plt.figure(figsize=(7, 6))
sns.barplot(
    data=top_go,
    x="minus_log10_fdr",
    y="Term"
)
plt.xlabel("-log10 adjusted p-value")
plt.ylabel("")
plt.title("GO enrichment of high-alpha genes")
plt.tight_layout()
plt.show()

In [ ]:
genes = {}

for term in sig_go["Term"].values:
     genes[term] = go_df.loc[
        go_df["Term"] == term,
        "Genes"
    ].iloc[0].split(";")

In [ ]:
for term, gs in genes.items():

    print(term)

    # keep only genes present in the dataset
    gs = [g for g in gs if g in adata.var_names]

    if len(gs) == 0:
        continue

    fig = sc.pl.spatial(
        adata,
        color=gs,
        cmap="viridis",
        ncols=3,
        spot_size=1,
        return_fig=True,
        layer="lognorm",
    )

The enrichment of the collagen-containing extracellular matrix term is particularly consistent with the biology of metastatic melanoma. Genes such as LRRC15, ELN, LTBP1, MFAP5, NID1, COL5A3, COL7A1, PXDN, and ADAMTSL4 are characteristic of extracellular matrix remodeling, activated fibroblasts, basement membrane organization, and vascular-associated stromal compartments. In stage III melanoma, metastatic invasion into lymph nodes is accompanied by substantial remodeling of the stromal microenvironment through fibroblast activation, collagen deposition, angiogenesis, and extracellular matrix reorganization. These processes are highly localized within the tissue, making them natural candidates for elevated gene-specific dispersion.

The enrichment of the muscle myosin complex initially appears surprising because lymphoid tissue does not contain skeletal muscle. However, the enrichment is driven primarily by MYH11 and MYL9, which are well-established markers of smooth muscle cells, pericytes, vascular-associated cells, and activated myofibroblasts rather than skeletal muscle. These contractile stromal populations are commonly observed around blood vessels and fibrotic regions within metastatic melanoma lesions, indicating that this enrichment likely reflects vascular and stromal remodeling rather than tissue contamination.

Several additional genes reinforce this interpretation. PDGFB is involved in pericyte recruitment and vascular maturation, F13A1 is frequently expressed in activated macrophages and stromal cells, whereas S100A8 and ADAMDEC1 are associated with inflammatory myeloid populations. Together, these genes indicate that the most overdispersed genes participate in processes occurring within spatially restricted stromal and immune niches rather than uniformly across the tissue.

## $\beta_j$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.histplot(beta, bins=50, kde=True, ax=axes[0])
axes[0].set_title("Distribution of spot-wise dispersion β")
axes[0].set_xlabel("beta")

probplot(beta.ravel(), dist="norm", plot=axes[1])
axes[1].set_title("QQ-plot of β")

plt.tight_layout()

plt.savefig(f"plots/K{K}_beta_distribution.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_beta_distribution.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
sc.pl.spatial(
    adata,
    color="beta",
    spot_size=1,
    cmap="magma",
    vmin=-0.2,
    vmax=0.7 # Avoid outlier making the plot undistinguisable
)

plt.savefig(f"plots/K{K}_beta_spatial_map.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_beta_spatial_map.pdf", dpi=300, bbox_inches="tight")
plt.close()

### $\beta$ vs library size

In [ ]:
# Compute QC metrics if not already present
if "total_counts" not in adata.obs.columns or "n_genes_by_counts" not in adata.obs.columns:
    sc.pp.calculate_qc_metrics(adata, inplace=True, layer="counts")

qc_vars = ["total_counts", "n_genes_by_counts"]

for qc in qc_vars:
    plt.figure(figsize=(5, 4))
    sns.scatterplot(
        data=adata.obs,
        x=qc,
        y="beta",
        s=15,
        alpha=0.7
    )
    plt.xlabel(qc)
    plt.ylabel("beta")
    plt.title(f"β vs {qc}")
    plt.tight_layout()

    plt.savefig(f"plots/K{K}_beta_{qc}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"plots/K{K}_beta_{qc}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    rho, pval = spearmanr(
        adata.obs[qc],
        adata.obs["beta"],
        nan_policy="omit"
    )

    print(f"Spearman beta vs {qc}: rho={rho:.3f}, p={pval:.3e}")

Negative because we are plotting the inverse dispersion. Still, **this low correlation (-0.32) demonstrates that $\beta$ is not merely technical, but captures biology**

### $\beta$ distribution per cell type

In [ ]:
plt.figure(figsize=(6, max(4, 0.3 * len(moran_df))))
ax = sns.boxplot(
    data=adata.obs,
    y="snmf_component",
    x="beta",
    palette=PALETTE
)

ax.set_xlim(-0.2, 0.8)

plt.xlabel("Beta")
plt.ylabel("Cell type")
plt.tight_layout()

plt.savefig(f"plots/K{K}_beta_per_celltype.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_beta_per_celltype.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
for ct in SIGNATURES.keys():
    print(f"Spearman correlation between '{ct}' proportion and beta: {spearmanr(adata.obs[ct], adata.obs['beta']).statistic:.3f}")

In [ ]:
adata.obs[ct]

### $\beta$ enrichment at component boundaries

In [ ]:
coords = adata.obs[["array_row", "array_col"]].to_numpy() if {"array_row", "array_col"}.issubset(adata.obs.columns) else adata.obsm["spatial"]
tree = cKDTree(coords)
_, nn = tree.query(coords, k=min(7, adata.n_obs))

dominant = adata.obs["snmf_component"].astype(str).to_numpy()
P = snmf_proportions.to_numpy()

boundary_score = []
prop_gradient = []

for i, neigh in enumerate(nn):
    neigh = [j for j in neigh if j != i]
    boundary_score.append(np.mean(dominant[neigh] != dominant[i]))
    prop_gradient.append(np.mean(np.abs(P[neigh] - P[i]).sum(axis=1) / 2))

adata.obs["component_boundary_score"] = boundary_score
adata.obs["component_prop_gradient"] = prop_gradient

boundary_mask = adata.obs["component_boundary_score"] > 0
print(compare_by_mask("beta", boundary_mask))
print("rho boundary vs beta:", spearmanr(adata.obs["component_boundary_score"], adata.obs["beta"]).statistic)
print("rho prop-gradient vs beta:", spearmanr(adata.obs["component_prop_gradient"], adata.obs["beta"]).statistic)

sc.pl.spatial(
    adata,
    color=["snmf_component", "component_boundary_score", "component_prop_gradient", "beta"],
    spot_size=1,
    frameon=False,
    cmap="magma",
    ncols=3,
    show=False,
)

plt.show()

# Antigen-presentation heterogeneity in tumor-dominant spots

In [ ]:
ANTIGEN_GENES = ["B2M", "CALR", "NLRC5", "PSMB9", "PSME1", "PSME3", "RFX5", "HSP90AB1"]

add_expr_score(ANTIGEN_GENES, "antigen_expr", layer="lognorm")
add_phi_score(ANTIGEN_GENES, "antigen_beta")

TUMOR_COMPONENTS = [c for c in SIGNATURES if "Tumor" in c]
adata.obs["tumor_score"] = adata.obs[TUMOR_COMPONENTS].sum(axis=1) if TUMOR_COMPONENTS else 1.0
tumor_mask = adata.obs["tumor_score"] >= adata.obs["tumor_score"].quantile(0.8)

stats_expr = compare_by_mask("antigen_expr", tumor_mask)
stats_beta = compare_by_mask("antigen_beta", tumor_mask)
print(stats_expr)
print(stats_beta)
print("rho tumor score vs antigen beta:", spearmanr(adata.obs["tumor_score"], adata.obs["antigen_beta"]).statistic)

sc.pl.spatial(
    adata,
    color=["tumor_score", "antigen_expr"],
    spot_size=1,
    frameon=False,
    cmap="magma",
    ncols=3,
    show=False,
)

plt.savefig(f"plots/K{K}_tumor_score_antigen_presentation.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_tumor_score_antigen_presentation.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Label tumor vs non-tumor spots
adata.obs["tumor_group"] = "Non-tumor"
adata.obs.loc[tumor_mask, "tumor_group"] = "Tumor"

# Plot beta values by tumor status
plot_df = adata.obs[["antigen_beta", "tumor_group"]].copy()
plot_df = plot_df.dropna()

plt.figure(figsize=(5, 4))
ax = sns.boxplot(
    data=plot_df,
    x="tumor_group",
    y="antigen_beta",
    hue="tumor_group",
    order=["Non-tumor", "Tumor"],
    # showfliers=False,
)

sns.stripplot(
    data=plot_df,
    x="tumor_group",
    y="antigen_beta",
    order=["Non-tumor", "Tumor"],
    color="black",
    alpha=0.25,
    size=2,
    jitter=0.25,
)

ax.set_ylim(0.1,1.0)

p_value = stats_beta["p"]
effect_size = stats_beta["effect_size_r"]

x1, x2 = 0, 1
y = 0.86
h = 0.015

ax.plot(
    [x1, x1, x2, x2],
    [y, y + h, y + h, y],
    color="black",
    linewidth=1.2,
)

ax.text(
    (x1 + x2) / 2,
    y + h + 0.008,
    f"p = {p_value:.2e}\n$r_{{rb}}$ = {effect_size:.2f}",
    ha="center",
    va="bottom",
)

plt.xlabel("")
plt.ylabel("Avg. dispersion accross\nantigen-presentation genes")
plt.tight_layout()

plt.savefig(f"plots/K{K}_beta_antigen_tumor_vs_nontumor.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_beta_antigen_tumor_vs_nontumor.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
palette = {
    "Tumor": sns.color_palette("tab20")[0],
    "Non-tumor": "#ededed"
}

fig = sc.pl.spatial(
    adata, 
    color="tumor_group",
    spot_size=1,
    frameon=False,
    return_fig=True,
    show=True,
    palette=palette,
)

fig.tight_layout()

fig.savefig(f"plots/K{K}_tumor_region.png", dpi=300, bbox_inches="tight")
fig.savefig(f"plots/K{K}_tumor_region.pdf", bbox_inches="tight")

plt.close(fig)

In [ ]:
adata.obs["tumor_center_mask"] = adata.obs["Tumor cells (center)"] >= adata.obs["Tumor cells (center)"].quantile(0.75)
adata.obs["tumor_border_mask"] = adata.obs["Tumor cells (border)"] >= adata.obs["Tumor cells (border)"].quantile(0.75)

x = adata.obs.loc[adata.obs["tumor_center_mask"], "antigen_expr"].dropna()
y = adata.obs.loc[adata.obs["tumor_border_mask"], "antigen_expr"].dropna()
p = mannwhitneyu(x, y, alternative="two-sided").pvalue if len(x) and len(y) else np.nan
print(f"""
    Test for 'antigen expression'
    Group 1: tumor center ({len(x)} spots)
    Group 2: tumor border ({len(y)} spots)

    Median 1: {x.median()}
    Median 2: {y.median()}

    P-value: {p}
""")

x = adata.obs.loc[adata.obs["tumor_center_mask"], "antigen_beta"].dropna()
y = adata.obs.loc[adata.obs["tumor_border_mask"], "antigen_beta"].dropna()
p = mannwhitneyu(x, y, alternative="two-sided").pvalue if len(x) and len(y) else np.nan
print(f"""
    Test for 'antigen beta'
    Group 1: tumor center ({len(x)} spots)
    Group 2: tumor border ({len(y)} spots)

    Median 1: {x.median()}
    Median 2: {y.median()}

    P-value: {p}
""")

# Label tumor vs non-tumor spots
adata.obs["tumor_group"] = "Tumor center"
adata.obs.loc[adata.obs["tumor_border_mask"], "tumor_group"] = "Tumor border"

# Plot beta values by tumor status
tumor_mask = adata.obs["tumor_center_mask"] | adata.obs["tumor_border_mask"]
plot_df = adata.obs.loc[tumor_mask, ["antigen_beta", "tumor_group"]].copy()
plot_df = plot_df.dropna()

plt.figure(figsize=(5, 4))
ax = sns.boxplot(
    data=plot_df,
    x="tumor_group",
    y="antigen_beta",
    hue="tumor_group",
    order=["Tumor center", "Tumor border"],
)

sns.stripplot(
    data=plot_df,
    x="tumor_group",
    y="antigen_beta",
    order=["Tumor center", "Tumor border"],
    color="black",
    alpha=0.25,
    size=2,
    jitter=0.25,
)

ax.set_ylim(0.1,0.9)

x1, x2 = 0, 1
y = 0.76
h = 0.015

ax.plot(
    [x1, x1, x2, x2],
    [y, y + h, y + h, y],
    color="black",
    linewidth=1.2,
)

ax.text(
    (x1 + x2) / 2,
    y + h + 0.01,
    f"p = {p:.2e}",
    ha="center",
    va="bottom",
)

plt.xlabel("")
plt.ylabel("Avg. dispersion accross\nantigen presentation genes")
plt.tight_layout()

plt.savefig(f"plots/K{K}_beta_antigen_tumor_center_vs_border.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_beta_antigen_tumor_center_vs_border.pdf", dpi=300, bbox_inches="tight")
plt.show()

# Transition area

In [ ]:
TRANSITION_AREA_EXPRESSED_GENES = ["FTL","B2M","APOE","HLA-A","HLA-B","HLA-C"]

In [ ]:
fig = sc.pl.spatial(
    adata,
    color=TRANSITION_AREA_EXPRESSED_GENES,
    spot_size=1,
    frameon=False,
    cmap="magma",
    ncols=len(TRANSITION_AREA_EXPRESSED_GENES),  # all in one row
    return_fig=True,
    show=True,
    layer="lognorm",
)

# Optional: resize to fit many genes
fig.set_size_inches(4 * len(TRANSITION_AREA_EXPRESSED_GENES), 4)
fig.tight_layout()

# Save to PNG and PDF
fig.savefig(f"plots/K{K}_V_transition_genes_spatial.png", dpi=300, bbox_inches="tight")
fig.savefig(f"plots/K{K}_V_transition_genes_spatial.pdf", bbox_inches="tight")

plt.close(fig)

In [ ]:
fig = sc.pl.spatial(
    adata,
    color=TRANSITION_AREA_EXPRESSED_GENES,
    spot_size=1,
    frameon=False,
    cmap="magma",
    ncols=len(TRANSITION_AREA_EXPRESSED_GENES),  # all in one row
    return_fig=True,
    show=True
)

# Optional: resize to fit many genes
fig.set_size_inches(4 * len(TRANSITION_AREA_EXPRESSED_GENES), 4)
fig.tight_layout()

# Save to PNG and PDF
fig.savefig(f"plots/K{K}_WHS_transition_genes_spatial.png", dpi=300, bbox_inches="tight")
fig.savefig(f"plots/K{K}_WHS_transition_genes_spatial.pdf", bbox_inches="tight")

plt.close(fig)

### Observed $V$

In [ ]:
add_expr_score(TRANSITION_AREA_EXPRESSED_GENES, "transition_area_score_V", layer="lognorm")

# Plot single spatial score
fig = sc.pl.spatial(
    adata,
    color="transition_area_score_V",
    spot_size=1,
    frameon=False,
    cmap="magma",
    title="Transition area score",
    return_fig=True,
    show=True
)

# Optional: resize figure for better visibility
fig.set_size_inches(6, 6)

fig.savefig(f"plots/K{K}_V_transition_score.png", dpi=300, bbox_inches="tight")
fig.savefig(f"plots/K{K}_V_transition_score.pdf", bbox_inches="tight")
fig.tight_layout()

In [ ]:
adata.obs["is_in_transition_area_V"] = np.where(
    adata.obs["transition_area_score_V"] > adata.obs["transition_area_score_V"].quantile(0.75),
    "in_transition",
    "out_of_transition"  # use a string instead of None
)

# Optional: make "out_of_transition" white
palette = {
    "in_transition": sns.color_palette("tab20")[0],
    "out_of_transition": "#ededed"
}

fig = sc.pl.spatial(
    adata,
    color="is_in_transition_area_V",
    spot_size=1,
    frameon=False,
    palette=palette,
    legend_loc="none",
    title="Transition Area",
    return_fig=True,
    show=True
)

# Optional: resize
fig.set_size_inches(6, 6)

fig.savefig(f"plots/K{K}_V_transition_mask.png", dpi=300, bbox_inches="tight")
fig.savefig(f"plots/K{K}_V_transition_mask.pdf", bbox_inches="tight")
fig.tight_layout()

### Inferred $\hat{V} = WHS$

In [ ]:
add_expr_score(TRANSITION_AREA_EXPRESSED_GENES, "transition_area_score_WHS")

# Plot single spatial score
fig = sc.pl.spatial(
    adata,
    color="transition_area_score_WHS",
    spot_size=1,
    frameon=False,
    cmap="magma",
    title="Transition area score",
    return_fig=True,
    show=True
)

# Optional: resize figure for better visibility
fig.set_size_inches(6, 6)
fig.tight_layout()

# Save both PNG and PDF
fig.savefig(f"plots/K{K}_WHS_transition_score.png", dpi=300, bbox_inches="tight")
fig.savefig(f"plots/K{K}_WHS_transition_score.pdf", bbox_inches="tight")
plt.close(fig)

In [ ]:
adata.obs["is_in_transition_area_WHS"] = np.where(
    adata.obs["transition_area_score_WHS"] > adata.obs["transition_area_score_WHS"].quantile(0.75),
    "in_transition",
    "out_of_transition"  # use a string instead of None
)

# Optional: make "out_of_transition" white
palette = {
    "in_transition": sns.color_palette("tab20")[0],
    "out_of_transition": "#ededed"
}

fig = sc.pl.spatial(
    adata,
    color="is_in_transition_area_WHS",
    spot_size=1,
    frameon=False,
    palette=palette,
    legend_loc="none",
    title="Transition Area",
    return_fig=True,
    show=True
)

# Optional: resize
fig.set_size_inches(6, 6)
fig.tight_layout()

# Save figure
fig.savefig(f"plots/K{K}_WHS_transition_mask.png", dpi=300, bbox_inches="tight")
fig.savefig(f"plots/K{K}_WHS_transition_mask.pdf", bbox_inches="tight")

plt.close(fig)

In [ ]:
for component in SIGNATURES.keys():
    rho, pval = spearmanr(adata.obs.loc[adata.obs["is_in_transition_area_WHS"] == "in_transition",component].values, adata.obs.loc[adata.obs["is_in_transition_area_WHS"] == "in_transition","transition_area_score_WHS"].values)
    print(f"Spearman correlation between '{component}' component and transition area score: {rho:.4f} (pval {pval:.4f})")

In [ ]:
from scipy.stats import linregress

mask = adata.obs["is_in_transition_area_WHS"] == "in_transition"

for component in SIGNATURES.keys():
    x = adata.obs.loc[mask, component].to_numpy()
    y = adata.obs.loc[mask, "transition_area_score_WHS"].to_numpy()

    # Remove missing or infinite values
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 2 or np.all(x == x[0]):
        print(f"Skipping {component}: insufficient variation")
        continue

    slope, intercept, r, p, _ = linregress(x, y)

    fig, ax = plt.subplots(figsize=(4, 4))

    ax.scatter(x, y, s=5, alpha=0.5)

    xx = np.linspace(x.min(), x.max(), 100)
    ax.plot(xx, slope * xx + intercept, color="red", lw=2)

    ax.set_xlabel(component)
    ax.set_ylabel("Transition area score")
    ax.set_title(f"$R$ = {r:.2f}, p = {p:.2e}")

    fig.tight_layout()

    safe_component = component.replace(" ", "_").replace("/", "_")

    fig.savefig(
        f"plots/K{K}_{safe_component}_corr.pdf",
        bbox_inches="tight"
    )
    fig.savefig(
        f"plots/K{K}_{safe_component}_corr.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

In [ ]:
from scipy.stats import entropy

adata.obs["entropy"] = entropy(snmf_proportions, axis=1)
rho, pval = spearmanr(adata.obs["entropy"].values, adata.obs["transition_area_score_WHS"].values)
print(f"Spearman correlation between proportions entropy and transition area: {rho:.4f} (pval {pval:.4f})")

# Residual analysis

In [ ]:
X = adata.layers["counts"]
Mu = adata.layers["inferred_counts"]

eps = 1e-10

Mu = np.maximum(Mu, eps)
Phi = np.maximum(phi, eps)

Var = Mu + Phi * Mu**2

pearson_residuals = (X - Mu) / np.sqrt(Var).to_numpy()

adata.layers["residual"] = pearson_residuals

In [ ]:
adata.obs["mean_residual"] = np.mean(pearson_residuals, axis=1)
sc.pl.spatial(
    adata,
    color="mean_residual",
    spot_size=1,
    frameon=False,
    cmap="magma",
    title="Mean residual",
)

plt.savefig(f"plots/K{K}_residuals.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_residuals.pdf", bbox_inches="tight")
plt.close()

The model presents a weaker fit in the tumor (center) region, where the $\beta$ is lower (dispersion is higher).

# Unannotated stroma region

In [ ]:
adata.obs["unannotated_stroma"] = (
    (adata.obs["Stroma"] >= adata.obs["Stroma"].quantile(0.75)) &
    (adata.obs["Tumor cells (center)"] >= adata.obs["Tumor cells (center)"].quantile(0.5))
)

sc.pl.spatial(
    adata,
    color="unannotated_stroma",
    spot_size=1,
    frameon=False,
    title=None
)

In [ ]:
adata.obs["histology_stroma"] = (
    (adata.obs["Stroma"] >= adata.obs["Stroma"].quantile(0.7)) &
    (adata.obs["Lymphoid"] >= adata.obs["Lymphoid"].quantile(0.4))
)

sc.pl.spatial(
    adata,
    color="histology_stroma",
    spot_size=1,
    frameon=False,
    title=None
)

In [ ]:
adata.obs["stroma_comparison"] = "Other"

adata.obs.loc[
    adata.obs["histology_stroma"],
    "stroma_comparison"
] = "Histology stroma"

adata.obs.loc[
    adata.obs["unannotated_stroma"],
    "stroma_comparison"
] = "Unannotated stroma"

adata.obs["stroma_comparison"] = pd.Categorical(
    adata.obs["stroma_comparison"],
    categories=[
        "Histology stroma",
        "Unannotated stroma",
        "Other"
    ]
)

sc.pl.spatial(
    adata,
    color="stroma_comparison",
    spot_size=1,
    frameon=False,
    title=None
)

plt.savefig(f"plots/K{K}_stroma_mask.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_stroma_mask.pdf", bbox_inches="tight")
plt.close()

In [ ]:
X = adata.layers["lognorm"]

if sparse.issparse(X):
    X = X.toarray()

hist = X[adata.obs["histology_stroma"]]
unann = X[adata.obs["unannotated_stroma"]]

hist_mean = hist.mean(axis=0)
unann_mean = unann.mean(axis=0)

r, _ = pearsonr(hist_mean, unann_mean)

print(f"Pearson correlation between mismatched and histology stroma = {r:.3f}")

adata.obs["melanoma"] = (
    adata.obs["snmf_component"] == "Tumor cells (center)"
)

melanoma = X[adata.obs["melanoma"]]
melanoma_mean = melanoma.mean(axis=0)

r, _ = pearsonr(hist_mean, melanoma_mean)

print(f"Pearson correlation between mismatched stroma and melanoma = {r:.3f}")

In [ ]:
adata_stroma = adata[
    adata.obs["stroma_comparison"].isin(
        ["Histology stroma", "Unannotated stroma"]
    )
].copy()

adata_stroma.obs["stroma_comparison"] = (
    adata_stroma.obs["stroma_comparison"]
    .cat.remove_unused_categories()
)

sc.tl.rank_genes_groups(
    adata_stroma,
    groupby="stroma_comparison",
    groups=["Unannotated stroma"],
    reference="Histology stroma",
    method="wilcoxon",
    layer="lognorm",
    use_raw=False,
    pts=True,
    key_added="unannotated_vs_histology_stroma"
)

In [ ]:
de = sc.get.rank_genes_groups_df(
    adata_stroma,
    group="Unannotated stroma",
    key="unannotated_vs_histology_stroma"
)

de = de.sort_values("pvals_adj")
de.head(30)

In [ ]:
sc.pl.rank_genes_groups(
    adata_stroma,
    key="unannotated_vs_histology_stroma",
    n_genes=25,
    sharey=False
)

plt.savefig(f"plots/K{K}_mismatched_vs_hist_stroma_de_genes.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_mismatched_vs_hist_stroma_de_genes.pdf", bbox_inches="tight")
plt.close()

In [ ]:
de = de.drop_duplicates(subset="names")

# Ranking for preranked GSEA
ranking = (
    de[["names", "scores"]]
    .dropna()
    .sort_values("scores", ascending=False)
)

pre_res = gp.prerank(
    rnk=ranking,
    gene_sets="MSigDB_Hallmark_2020",
    threads=4,
    min_size=10,
    max_size=500,
    permutation_num=1000,
    seed=42,
    outdir=None
)

results = pre_res.res2d
results = results.sort_values("FDR q-val")
results.head(20)

In [ ]:
gp.barplot(
    results,
    column="FDR q-val",
    top_term=20,
    figsize=(7,6)
)

plt.savefig(f"plots/K{K}_mismatched_vs_hist_stroma_gsea_barplot.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_mismatched_vs_hist_stroma_gsea_barplot.pdf", bbox_inches="tight")
plt.show()

In [ ]:
gp.dotplot(
    results,
    column="FDR q-val",
    top_term=20,
    figsize=(7,6)
)

plt.savefig(f"plots/K{K}_mismatched_vs_hist_stroma_gsea_dotplot.png", dpi=300, bbox_inches="tight")
plt.savefig(f"plots/K{K}_mismatched_vs_hist_stroma_gsea_dotplot.pdf", bbox_inches="tight")
plt.show()